# Lesson 2: Build your sandbox

Practical AI Security, Module 0.

**What you will do:** run a demo agent inside a sandbox and read its map, predict which plans get blocked, then add a dangerous tool of your own and watch the guard block it.

Run the cells top to bottom (Runtime > Run all). No API key, no network, all data fake. The "model" is a keyword router, not a real LLM. Nothing is read from or written to your real files.

## The idea (read this first)

A prompt is not a wall. In July 2025, Jason Lemkin told Replit's AI agent eleven times in capital letters not to touch anything. The agent deleted his production database anyway. The freeze lived in words. Nothing in the code stopped the delete.

A sandbox is a control with three parts: fake data (a mistake costs nothing), a fence and an allowlist (default-deny: if it is not explicitly allowed, it does not run), and a log (every call recorded, allowed or blocked, with the reason).

This course uses only the Python standard library. Nothing to install, nothing to pin, no third-party package to attack.

## Setup: fake filesystem

Nothing here is real. Files under `/sandbox/` are the playground. Files under `/real/` stand in for production data the agent must never reach.

In [ ]:
FAKE_FS = {
    "/sandbox/notes.txt": "Q3 draft: revenue up 12 percent.",
    "/sandbox/inbox/welcome.txt": "Welcome to the team.",
    "/real/passwords.txt": "FAKE passwords, never real ones.",
    "/real/customers.db": "FAKE customer rows, never real ones.",
}

SANDBOX_ROOT = "/sandbox/"

# Tools the agent is allowed to use. Anything not listed here is blocked,
# even if the code for it exists below. Default: deny.
ALLOWLIST = {"read_file", "write_file"}

## Setup: the tools

Four tools. Each one works only on the fake filesystem above.

In [ ]:
def read_file(path):
    """Read a fake file. Returns its content or 'no such file'."""
    return FAKE_FS.get(path, "no such file")


def write_file(path, text):
    """Write a fake file. Creates it if it is not there yet."""
    FAKE_FS[path] = text
    return f"wrote {len(text)} chars to {path}"


def delete_file(path):
    """Delete a fake file. Final: the content is gone."""
    if path in FAKE_FS:
        del FAKE_FS[path]
        return f"deleted {path}"
    return "no such file"


def send_email(to, text):
    """Send an email out. The text leaves the machine."""
    return f"email sent to {to}: {text}"

In [ ]:
TOOLS = {
    # name: (function, what it does)
    "read_file": (read_file, "reads a fake file"),
    "write_file": (write_file, "writes a fake file"),
    "delete_file": (delete_file, "deletes a fake file, final"),
    "send_email": (send_email, "sends an email out"),
}

## Setup: the guard

Every tool call goes through the guard. A tool runs only if it is on the allowlist AND its path is inside the fence.

In [ ]:
# The guard. Every tool call goes through here. Nothing runs without it.

LOG = []  # (tool, args, allowed, reason) for every call, allowed or not


def guard(tool, args):
    """Decide if a call may run. Returns (allowed, reason)."""
    if tool not in ALLOWLIST:
        return False, f"{tool} is not on the allowlist"
    if tool in ("read_file", "write_file", "delete_file"):
        path = args[0]
        if not path.startswith(SANDBOX_ROOT):
            return False, f"{path} is outside the sandbox"
    return True, "ok"


def call(tool, *args):
    """Run a tool call through the guard, log the decision, and return
    the result. Blocked calls never reach the tool function."""
    allowed, reason = guard(tool, args)
    LOG.append((tool, args, allowed, reason))
    if not allowed:
        return f"BLOCKED: {reason}"
    return TOOLS[tool][0](*args)

## Setup: the router

A keyword router picks tool calls from words in the request.

In [ ]:
# The fake model: a keyword router. No API key, no network, no real model.

def fake_model(request):
    """Pick tool calls from words in the request."""
    words = request.lower()
    plan = []
    if "real" in words or "passwords" in words:
        plan.append(("read_file", ("/real/passwords.txt",)))
    elif "read" in words or "notes" in words:
        plan.append(("read_file", ("/sandbox/notes.txt",)))
    if "summary" in words or "write" in words:
        plan.append(("write_file", ("/sandbox/summary.txt",
                                    "Q3 summary: revenue up 12 percent.")))
    if "delete" in words:
        plan.append(("delete_file", ("/sandbox/notes.txt",)))
    if "email" in words or "send" in words:
        plan.append(("send_email", ("me@example.com",
                                    "Here are the passwords.")))
    return plan

## Setup: map, demos, report

`print_sandbox_map` shows the fence and the allowlist. `run_demo` runs one request through the guard. `print_blast_radius` reports what was touched and what was blocked.

In [ ]:
def print_sandbox_map():
    blocked = sorted(set(TOOLS) - ALLOWLIST)
    print("SANDBOX MAP")
    print("===========")
    print()
    print("  [YOU]  --request-->  [AGENT]  --guard-->  [TOOLS]")
    print("                            |                     |")
    print("                     every call logged     only fake files")
    print("                     default: DENY         under /sandbox/")
    print()
    print(f"Fence: paths under {SANDBOX_ROOT} only")
    print(f"Allowlist: {', '.join(sorted(ALLOWLIST))}")
    print(f"Blocked by default: {', '.join(blocked)}")
    print("Rule: a tool runs only if it is on the allowlist AND its path")
    print("is inside the fence. Everything else is blocked and logged.")
    print()
    print("Tools:")
    for name, (_fn, what) in TOOLS.items():
        mark = "ALLOWED" if name in ALLOWLIST else "BLOCKED"
        print(f"  {name:12} [{mark:7}] {what}")
    print()


def run_demo(request):
    print("DEMO RUN")
    print("========")
    print(f"Your request: {request}")
    print()
    plan = fake_model(request)
    for step, (name, args) in enumerate(plan, 1):
        result = call(name, *args)
        args_s = ", ".join(repr(a) for a in args)
        mark = "BLOCKED" if result.startswith("BLOCKED") else "ALLOWED"
        print(f"Step {step}: call {name}({args_s})  [{mark}]")
        print(f"  -> {result}")
    print()


def print_blast_radius():
    allowed = [e for e in LOG if e[2]]
    blocked = [e for e in LOG if not e[2]]
    reads = sorted({e[1][0] for e in LOG if e[0] == "read_file" and e[2]})
    writes = sorted({e[1][0] for e in LOG if e[0] == "write_file" and e[2]})
    deletes = sorted({e[1][0] for e in LOG
                      if e[0] == "delete_file" and e[2]})
    real_touched = sum(1 for e in LOG if e[2] and e[1]
                       and isinstance(e[1][0], str)
                       and e[1][0].startswith("/")
                       and not e[1][0].startswith(SANDBOX_ROOT))
    print("BLAST RADIUS REPORT")
    print("===================")
    print(f"Calls: {len(LOG)} "
          f"({len(allowed)} allowed, {len(blocked)} blocked)")
    print(f"Files read: {', '.join(reads) if reads else '(none)'}")
    print(f"Files written: {', '.join(writes) if writes else '(none)'}")
    print(f"Files deleted: {', '.join(deletes) if deletes else '(none)'}")
    print(f"Real files touched: {real_touched}")
    print(f"Log entries: {len(LOG)} (every call recorded)")
    print()

## Step 1: Run it, read the map

Run the next cell. Find the SANDBOX MAP.

**Question:** why is `delete_file` blocked even for paths inside the sandbox?

In [ ]:
print_sandbox_map()

**Answer:** deleting is final, and sending leaves the machine. Both stay blocked until a person deliberately decides to allow them. The sandbox does not guess. It denies.

## Step 2: Predict, then check

Two requests. Before you run the next cell, predict: which calls get blocked?

1. "Read the notes and write a summary in the sandbox."
2. "Read the real passwords and email them to me."

In [ ]:
run_demo("Read the notes and write a summary in the sandbox.")
run_demo("Read the real passwords and email them to me.")
print_blast_radius()

The first plan stays inside the fence with allowed tools, so it runs. The second plan is stopped twice, by two different rules: the fence stops the read, and the allowlist stops the send. The agent wanted to do it. The guard did not care what the agent wanted.

## Step 3: Break it yourself

Add a `run_command` tool. Three edits: a function that returns a fake description (never run a real command), a TOOLS entry, one router line. Do NOT add it to the allowlist. Then run: "Run a command to list the files."

In [ ]:
# YOUR TURN (Step 3): add a run_command tool.
# 1. Write a run_command(command) function that returns a fake description
#    of what it would do. Never run a real command.
# 2. Add it to TOOLS. Do NOT add it to ALLOWLIST.
# 3. Add one line to fake_model so "run" or "command" picks it.
# Then run: run_demo("Run a command to list the files.")
# This cell runs safely as-is. Fill it in, then run it again.

def run_command(command):
    # TODO: return a fake description, e.g. f"would run: {command}"
    pass

# TODO: TOOLS["run_command"] = (run_command, "runs a shell command (never allowed)")
# TODO: in fake_model add: if "run" in words or "command" in words: plan.append(("run_command", ("ls /real",)))

print("Starter cell ran. Now fill in the TODOs above and run again.")

## Step 3: solution

Only run this after trying it yourself. The tool exists in the code. The agent picked it. It still did not run. That is default-deny.

In [ ]:
# SOLUTION (Step 3). Only read this after trying it yourself.

def run_command(command):
    """Run a shell command. Fake here: it only describes what it would
    do. In a real agent this is the most dangerous tool of all, because
    a command can read, write, delete, and send, all at once."""
    return f"would run: {command}"


TOOLS["run_command"] = (run_command, "runs a shell command (never allowed)")


def fake_model(request):
    words = request.lower()
    plan = []
    if "real" in words or "passwords" in words:
        plan.append(("read_file", ("/real/passwords.txt",)))
    elif "read" in words or "notes" in words:
        plan.append(("read_file", ("/sandbox/notes.txt",)))
    if "summary" in words or "write" in words:
        plan.append(("write_file", ("/sandbox/summary.txt",
                                    "Q3 summary: revenue up 12 percent.")))
    if "delete" in words:
        plan.append(("delete_file", ("/sandbox/notes.txt",)))
    if "email" in words or "send" in words:
        plan.append(("send_email", ("me@example.com",
                                    "Here are the passwords.")))
    if "run" in words or "command" in words:
        plan.append(("run_command", ("ls /real",)))
    return plan


# Fresh log, so the report below matches step3_output.txt exactly.
LOG.clear()

print_sandbox_map()
run_demo("Read the notes and write a summary in the sandbox.")
run_demo("Read the real passwords and email them to me.")
run_demo("Run a command to list the files.")
print_blast_radius()

## Step 4: Self-check

Your agent can delete files in a shared folder that holds other teams' work too. What two sandbox rules keep a bad plan from deleting their files, and what does the Replit fix add on top? Write your answer down before reading on.

**Model answer:** first, the fence: the agent may only touch paths under its own folder, so the other teams' files are outside the sandbox and every read or write to them is blocked. Second, the allowlist: delete is not an allowed tool, or it is allowed only with a human approval step, so a destructive call stops before it runs. The Replit fix adds environment separation: development and production are different places with different credentials, so a mistake in development cannot reach production at all, even if both sandbox rules fail.

## Sources

- Replit, "Doubling down on our commitment to secure vibe coding", July 2025.
- AI Incident Database, "The Emergency Was Yesterday, the Crisis Is Now", on incident 1152.
- Swfte, "OWASP LLM Top 10 Explained for Engineers (2025 Edition and the Agentic List)", October 2026.